# Classical references

Price a one-year European call with spot and strike both 100, volatility 20%,
and a 3% continuously compounded rate. Black–Scholes supplies the exact
price and Greeks under these inputs. Terminal Monte Carlo and a one-dimensional
PDE solve give two independent numerical checks.

In [1]:
import sys
from pathlib import Path

import numpy as np
from IPython.display import Markdown, display

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))

from bsde_pricing.black_scholes import BlackScholesReference
from bsde_pricing.finite_difference import FiniteDifferencePricer
from bsde_pricing.monte_carlo import MonteCarloPricer
from bsde_pricing.problems import vanilla_case

case = vanilla_case()
spot = case.initial_spots[0]
exact = BlackScholesReference(case.model, case.contract).evaluate(spot)
display(Markdown(
    f"| Price | Delta | Gamma |\n| ---: | ---: | ---: |\n"
    f"| {exact.call_price:.8f} | {exact.call_delta:.8f} | {exact.gamma:.8f} |"
))

| Price | Delta | Gamma |
| ---: | ---: | ---: |
| 9.41340338 | 0.59870633 | 0.01933341 |

Delta is the first spot derivative. A one-unit spot increase changes the
price by about 0.60 locally. Positive gamma measures how quickly that
delta increases with spot.

## Terminal Monte Carlo

Exact GBM sampling avoids a time-stepping error. For this stochastic call,
the price standard error is the sample standard deviation of discounted
payoffs divided by $\sqrt n$. The three allocations below use nested draws.

In [2]:
mc = MonteCarloPricer(case.model, case.contract)
lines = ["| Paths | Price | Signed error | 95% half-width |", "| ---: | ---: | ---: | ---: |"]
for count in (4_096, 16_384, 65_536):
    price = mc.estimate(case.initial_spots, n_paths=count, rng=np.random.default_rng(2026))
    lines.append(
        f"| {count:,} | {price.price_estimate:.6f} | {price.price_estimate-exact.call_price:+.6f} "
        f"| {price.confidence_high-price.price_estimate:.6f} |"
    )
display(Markdown("\n".join(lines)))

| Paths | Price | Signed error | 95% half-width |
| ---: | ---: | ---: | ---: |
| 4,096 | 9.323840 | -0.089564 | 0.427884 |
| 16,384 | 9.438725 | +0.025322 | 0.214572 |
| 65,536 | 9.415953 | +0.002550 | 0.107675 |

Quadrupling the path count roughly halves the interval width. The signed
errors depend on the realized draws. A single allocation therefore
measures sampling precision only through its standard error.

## Spot delta by two Monte Carlo estimators

The pathwise estimator differentiates the discounted payoff away from
the strike. The central bump prices the up and down states with the same
normals. Their paired difference largely cancels simulation noise.

In [3]:
risk = mc.estimate_delta(case.initial_spots, n_paths=65_536, bump_fraction=0.001,
                         rng=np.random.default_rng(2026))
lines = ["| Estimator | Delta | Signed error | Standard error |", "| --- | ---: | ---: | ---: |"]
for label, stat in (("Pathwise", risk.pathwise), ("Central bump", risk.central_bump)):
    lines.append(f"| {label} | {stat.estimate:.6f} | {stat.estimate-exact.call_delta:+.6f} "
                 f"| {stat.standard_error:.6f} |")
display(Markdown("\n".join(lines)))

| Estimator | Delta | Signed error | Standard error |
| --- | ---: | ---: | ---: |
| Pathwise | 0.600217 | +0.001511 | 0.002285 |
| Central bump | 0.600267 | +0.001561 | 0.002283 |

Both estimates use the same underlying sample and lie close to one
another. Their common deviation from the exact delta should be read
relative to their sampling uncertainty. The bump also has a finite-step
approximation error.

## Finite-difference refinement

Crank–Nicolson evolves the payoff backwards. Two startup intervals use
backward-Euler half steps to damp the kink. The domain stays fixed at
400 while the number of space and time intervals doubles.

In [4]:
fd = FiniteDifferencePricer(case.model, case.contract)
lines = ["| Intervals per axis | Price error | Delta error | Gamma error |",
         "| ---: | ---: | ---: | ---: |"]
for size in (100, 200, 400, 800):
    value = fd.solve(s_max=400, n_space=size, n_time=size).evaluate(spot)
    lines.append(f"| {size} | {value.call_price-exact.call_price:+.3e} "
                 f"| {value.call_delta-exact.call_delta:+.3e} | {value.gamma-exact.gamma:+.3e} |")
display(Markdown("\n".join(lines)))

| Intervals per axis | Price error | Delta error | Gamma error |
| ---: | ---: | ---: | ---: |
| 100 | -3.953e-02 | -8.753e-04 | +1.115e-04 |
| 200 | -9.827e-03 | -2.169e-04 | +2.741e-05 |
| 400 | -2.453e-03 | -5.409e-05 | +6.825e-06 |
| 800 | -6.131e-04 | -1.352e-05 | +1.705e-06 |

Each doubling cuts the three errors to roughly one-quarter.
These are deterministic differences from Black–Scholes. The study
combines spatial and temporal error at a fixed boundary and at a spot
that lies on every grid.